In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
import os
sys.path.append(os.path.abspath("") + "../../../")
from nnaero.geometry import *

## Inference

In [4]:

airfoils = Airfoil(
    names=["naca4412","naca4412", "naca9412"]
)

kulfan_parameters = kulfan.parameterize(airfoils)


### 1 to 1 varying of airfoils and parameters

In [5]:
from nnaero.models.surrogate_models import neuralfoil

alpha = np.array([5,10,5])
Re = np.array([1e6, 1e6, 1e6])
mach=0.2
n_crit=9
xtr_upper=0.10
xtr_lower=1.00

aero_torch_batched = neuralfoil.get_aero_from_kulfan(
    kulfan_parameters,
    alpha,
    Re,
    n_crit,
    xtr_upper,
    xtr_lower,
    model_size="medium",
    device="cpu",
)

In [6]:
aero_torch_batched

{'analysis_confidence': array([0.94980246, 0.9317807 , 0.9527096 ], dtype=float32),
 'CL': array([0.9693881, 1.415195 , 1.3562107], dtype=float32),
 'CD': array([0.01098982, 0.01696916, 0.01753247], dtype=float32),
 'CM': array([-0.09387028, -0.07655275, -0.19029817], dtype=float32),
 'Top_Xtr': array([0.08996906, 0.03270481, 0.0842289 ], dtype=float32),
 'Bot_Xtr': array([1., 1., 1.], dtype=float32),
 'upper_bl_theta_0': array([7.292734e-05, 8.490464e-05, 7.611029e-05], dtype=float32),
 'upper_bl_H_0': array([2.488523 , 2.8216267, 2.4314013], dtype=float32),
 'upper_bl_ue/vinf_0': array([1.5101463, 2.1358678, 1.3062598], dtype=float32),
 'lower_bl_theta_0': array([5.4412492e-05, 8.2500483e-05, 6.0036462e-05], dtype=float32),
 'lower_bl_H_0': array([2.3011048, 2.2111492, 2.292131 ], dtype=float32),
 'lower_bl_ue/vinf_0': array([-0.43451   ,  0.13669515, -0.5476353 ], dtype=float32),
 'upper_bl_theta_1': array([0.0001195 , 0.00018745, 0.00011235], dtype=float32),
 'upper_bl_H_1': array(

In [7]:
aero_torch_batched["CD"].shape, kulfan_parameters["lower_weights"].shape, alpha.shape

((3,), (3, 8), (3,))

### Broadcasting parameters

In [8]:
from nnaero.models.surrogate_models import neuralfoil

alpha = np.array([5,10,5,10])
Re = np.array([1e6, 1e6, 1e6,1e6])
mach=0.2
n_crit=9
xtr_upper=0.10
xtr_lower=1.00

aero_torch_batched = neuralfoil.get_aero_from_kulfan(
    kulfan_parameters,
    alpha,
    Re,
    n_crit,
    xtr_upper,
    xtr_lower,
    model_size="medium",
    device="cpu",
)

In [9]:
aero_torch_batched["CD"].shape, len(airfoils), alpha.shape

((3, 4), 3, (4,))

### Surrogate + modifications

In [15]:
alpha = np.array([5,10,5,25])
Re = np.array([1e6, 1e6, 1e6,1e8])
mach=np.array([0.0,0.2,0.4,0.5])

aero_modified = neuralfoil.get_aero(
    kulfan_parameters,
    alpha,
    Re,
    mach,
    n_crit,
    xtr_upper,
    xtr_lower,
    max_thickness=airfoils.max_thickness(),
    model_size="medium",
    device="cpu",
)

In [16]:
aero_modified

{'analysis_confidence': array([[0.94980216, 0.93178034, 0.94980216, 0.12281344],
        [0.94980216, 0.93178034, 0.94980216, 0.12281344],
        [0.9527098 , 0.93256265, 0.9527098 , 0.06139278]], dtype=float32),
 'CL': array([[0.9650192 , 1.43644393, 1.04709924, 1.22566948],
        [0.9650192 , 1.43644393, 1.04709924, 1.22566948],
        [1.35009897, 1.72370116, 1.4649321 , 1.27872974]]),
 'CD': array([[0.01098982, 0.01696917, 0.01098982, 0.18714834],
        [0.01098982, 0.01696917, 0.01098982, 0.18714834],
        [0.01753248, 0.02956471, 0.01753248, 0.2012674 ]]),
 'CM': array([[-0.21373531, -0.2548982 , -0.23190468, -0.31408017],
        [-0.21373531, -0.2548982 , -0.23190468, -0.31408017],
        [-0.35775167, -0.37953968, -0.38816457, -0.41942379]]),
 'Cpmin': array([[ -1.38287453,  -3.61541279,  -1.50049874, -11.62770184],
        [ -1.38287453,  -3.61541279,  -1.50049874, -11.62770184],
        [ -1.64925515,  -2.40806873,  -1.7895371 ,  -8.82454137]]),
 'Top_Xtr': array([